# 04 - Feature Engineering
## Credit Risk Analysis — Give Me Some Credit

**Objetivo de este notebook:**
- Crear nuevas variables que aporten información adicional al modelo
- Codificar variables categóricas si es necesario
- Escalar las variables numéricas
- Guardar el dataset final listo para el modelado

In [1]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Caragamos dataset limpio
df= pd.read_csv('../data/processed/cs-training-clean.csv')
df.head()

,seriousdlqin2yrs,revolvingutilizationofunsecuredlines,age,numberoftime30_59dayspastduenotworse,debtratio,monthlyincome,numberofopencreditlinesandloans,numberoftimes90dayslate,numberrealestateloansorlines,numberoftime60_89dayspastduenotworse,numberofdependents
0,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,0,0.907239,49,1,0.024926,12627.5,7,0,1,0,0.0


In [5]:
# Creamos 'total_delays" sumando todas las columnas de retrasos
total_delays= df[['numberoftime30_59dayspastduenotworse', 'numberoftime60_89dayspastduenotworse', 'numberoftimes90dayslate']].sum(axis=1)
df['total_delays']= total_delays
df['total_delays'].head()

0    2
1    0
2    2
3    0
4    1
Name: total_delays, dtype: int64

In [6]:
# Ahora creamos 'income_per_dependent' dividiendo 'monthlyincome' por 'numberofdependents'
income_per_dependent= df['monthlyincome'] / df['numberofdependents'].replace(0, 1) # Reemplazamos 0 por 1 para evitar división por cero
df['income_per_dependent']= income_per_dependent
df['income_per_dependent'].head()

0     4560.0
1     2600.0
2     3042.0
3     3300.0
4    12627.5
Name: income_per_dependent, dtype: float64

In [7]:
# Ahora vamos a crear 'credit_utilizaton_risk' combinando 'revolvingutilizationofunsecuredlines' y 'debtratio' para identificar clientes con alto riesgo de crédito
df['credit_utilization_risk'] = np.where((df['revolvingutilizationofunsecuredlines'] + df['debtratio']) > 1, 1, 0)
df['credit_utilization_risk'].value_counts()

credit_utilization_risk
0    87726
1    62005
Name: count, dtype: int64

In [8]:
# Por último, creamos 'financial_margin' multiplicando 'monthlyincome'*(1-debtratio) para estimar el margen financiero de cada cliente
df['financial_margin'] = df['monthlyincome'] * (1 - df['debtratio'])
df['financial_margin'].head()

0     1796.802984
1     2283.121877
2     2783.085113
3     3181.036049
4    12312.750786
Name: financial_margin, dtype: float64

In [9]:
#Comprobamos las nuevas variables
df.columns

Index(['seriousdlqin2yrs', 'revolvingutilizationofunsecuredlines', 'age',
       'numberoftime30_59dayspastduenotworse', 'debtratio', 'monthlyincome',
       'numberofopencreditlinesandloans', 'numberoftimes90dayslate',
       'numberrealestateloansorlines', 'numberoftime60_89dayspastduenotworse',
       'numberofdependents', 'total_delays', 'income_per_dependent',
       'credit_utilization_risk', 'financial_margin'],
      dtype='str')

## ¿Qué variables hemos creado y por qué?
- 'total_delays': Ha sido creada para poder saber el nº total de retrasos, para poder generar un indicador global del historial de pago de cada usuario
- 'income_per_dependent': Creada para poder ver los ingresos mensuales divididos entre en nº de personas dependientes, esto lo utilzaremos para saber realmente la carga financiera por persona, y poder ver como se distribuye el gasto por hogar, y con ello poder ver la correlación de impagos.
- 'credit_utilization_risk': La vamos a utilizar para poder comprobar el riesgo global de endeudamiento que tienen los usuarios, y poder predecir si un usuario tendrá dificultades para poder pagar, calculandolo con la suma de 'revolvingutilizationofunsecuredlines' y 'debtratio'; si la suma supera 1 se considera de alto riesgo.
- 'financial_margin': Creada para poder ver la capacidad de ahorro o margen financiero, como un indicador real de estabilidad. La fórmula para calcularlo es " monthlyincome x (1-debtratio).

In [10]:
# Gyuardamos el dataset con las nuevas variables
df.to_csv('../data/processed/cs-training-features.csv', index=False)
print("Dataset con nuevas variables guardado correctamente")
print(f'Shape final: {df.shape}')

Dataset con nuevas variables guardado correctamente
Shape final: (149731, 15)
